In [15]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.library.parameters import SeasonAll

# Get LeBron
all_players = players.get_players()
lebron = [p for p in all_players if p['full_name'] == 'LeBron James'][0]

print(f"Getting game log for {lebron['full_name']}")

game_log = playergamelog.PlayerGameLog(
    player_id=lebron['id'],
    season=SeasonAll.all
)

df = game_log.get_data_frames()[0]

print(f"\nTotal columns: {len(df.columns)}")
print("\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:3d}. {col}")

print("\nSample row:")
print(df.head(1).T)

Getting game log for LeBron James

Total columns: 27

Column names:
    1. SEASON_ID
    2. Player_ID
    3. Game_ID
    4. GAME_DATE
    5. MATCHUP
    6. WL
    7. MIN
    8. FGM
    9. FGA
   10. FG_PCT
   11. FG3M
   12. FG3A
   13. FG3_PCT
   14. FTM
   15. FTA
   16. FT_PCT
   17. OREB
   18. DREB
   19. REB
   20. AST
   21. STL
   22. BLK
   23. TOV
   24. PF
   25. PTS
   26. PLUS_MINUS
   27. VIDEO_AVAILABLE

Sample row:
                            0
SEASON_ID               22025
Player_ID                2544
Game_ID            0022500317
GAME_DATE        Dec 01, 2025
MATCHUP           LAL vs. PHX
WL                          L
MIN                        31
FGM                         3
FGA                        10
FG_PCT                    0.3
FG3M                        1
FG3A                        4
FG3_PCT                  0.25
FTM                         3
FTA                         4
FT_PCT                   0.75
OREB                        0
DREB                     

In [41]:
from nba_api.stats.endpoints import boxscoreadvancedv2
from nba_api.stats.library.http import NBAStatsHTTP


gameID = '0022500059'
print(f"Testing BoxScoreAdvancedV2 with game_id: {gameID}")
# boxscoreadvancedv2.BoxScoreAdvancedV2(
#     game_id=gameID,
# )

# endpoint = boxscoreadvancedv2.BoxScoreAdvancedV2(
#     game_id=gameID,
#     get_request=False
# )

# # 2) Manually send the request using NBAStatsHTTP
# http = NBAStatsHTTP()
# nba_response = http.send_api_request(
#     endpoint=endpoint.endpoint,
#     parameters=endpoint.parameters,
# )

# print(nba_response.get_dict())
# print(nba_response._response)

resp = boxscoreadvancedv2.BoxScoreAdvancedV2(
    game_id="0022500059",
    headers=headers,
    timeout=30
)

print(resp.player_stats.get_data_frame())

# resp = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=gameID)
# resp = resp.nba_response #.raw_json
# print(resp)
# dfs = advanced.get_data_frames()
# dfs

Testing BoxScoreAdvancedV2 with game_id: 0022500059


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [35]:
from nba_api.stats.endpoints import boxscoreadvancedv2
from nba_api.stats.library.http import NBAStatsHTTP
import json

test_game_id = '0022301148'

print(f"Testing BoxScoreAdvancedV2 with game_id: {test_game_id}")
print("=" * 80)

# Method 1: Try using the endpoint directly
print("\n1. Testing endpoint with get_request=False")
try:
    endpoint = boxscoreadvancedv2.BoxScoreAdvancedV2(
        game_id=test_game_id,
        get_request=False
    )
    print(f"  ✓ Created endpoint object")
    
    # Manually make the request
    print("\n2. Making manual API request...")
    endpoint.get_request()
    print(f"  ✓ Got response")
    
    # Check the response object
    print("\n3. Inspecting response...")
    if hasattr(endpoint, 'nba_response'):
        print(f"  Has nba_response: True")
        response = endpoint.nba_response
        
        # Try to get the raw JSON
        if hasattr(response, 'get_dict'):
            raw_dict = response.get_dict()
            print(f"  Response keys: {raw_dict.keys()}")
            
            # Check resultSets structure
            if 'resultSets' in raw_dict:
                print(f"\n  Number of resultSets: {len(raw_dict['resultSets'])}")
                for i, rs in enumerate(raw_dict['resultSets']):
                    print(f"    ResultSet {i}: {rs.get('name', 'unnamed')}")
                    if 'headers' in rs:
                        print(f"      Headers: {rs['headers'][:5]}...")
                    if 'rowSet' in rs:
                        print(f"      Rows: {len(rs['rowSet'])}")
            
            # Check for alternative keys
            if 'resultSet' in raw_dict:
                print(f"\n  Found 'resultSet' (singular)")
                
        # Try get_data_sets
        if hasattr(response, 'get_data_sets'):
            print("\n4. Trying get_data_sets()...")
            try:
                data_sets = response.get_data_sets()
                print(f"  ✓ Data sets: {data_sets.keys()}")
            except Exception as e:
                print(f"  ✗ get_data_sets() failed: {e}")
                
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

# Method 2: Try with a different game ID (maybe this one is invalid?)
print("\n\n" + "=" * 80)
print("Testing with a more recent game ID")
print("=" * 80)

# Try a game from early in 2023-24 season
test_game_id_2 = '0022300001'  # First game of 2023-24 season
print(f"\nTrying game_id: {test_game_id_2}")

try:
    advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=test_game_id_2)
    dfs = advanced.get_data_frames()
    print(f"  ✓ SUCCESS! Got {len(dfs)} dataframes")
    print(f"    PlayerStats shape: {dfs[0].shape}")
    print(f"    Columns: {list(dfs[0].columns)}")
except Exception as e:
    print(f"  ✗ Failed: {e}")

Testing BoxScoreAdvancedV2 with game_id: 0022301148

1. Testing endpoint with get_request=False
  ✓ Created endpoint object

2. Making manual API request...

❌ Error: 'resultSet'


Testing with a more recent game ID

Trying game_id: 0022300001
  ✗ Failed: 'resultSet'


Traceback (most recent call last):
  File "C:\Users\Chase\AppData\Local\Temp\ipykernel_12804\2649422562.py", line 21, in <module>
    endpoint.get_request()
  File "C:\Users\Chase\anaconda3\Lib\site-packages\nba_api\stats\endpoints\boxscoreadvancedv2.py", line 123, in get_request
    self.load_response()
  File "C:\Users\Chase\anaconda3\Lib\site-packages\nba_api\stats\endpoints\boxscoreadvancedv2.py", line 126, in load_response
    data_sets = self.nba_response.get_data_sets()
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Chase\anaconda3\Lib\site-packages\nba_api\stats\library\http.py", line 129, in get_data_sets
KeyError: 'resultSet'


In [37]:
from nba_api.stats.endpoints import boxscoreadvancedv2
import time

# Get a sample game ID from your test run
# Using one from the 2023-24 season
test_game_id = '0022301148'

print(f"Testing BoxScoreAdvancedV2 with game_id: {test_game_id}")
print("=" * 60)

advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=test_game_id)
advanced

Testing BoxScoreAdvancedV2 with game_id: 0022301148


KeyError: 'resultSet'

In [22]:
from nba_api.stats.endpoints import boxscoreadvancedv2
import time

# Get a sample game ID from your test run
# Using one from the 2023-24 season
test_game_id = '0022301148'

print(f"Testing BoxScoreAdvancedV2 with game_id: {test_game_id}")
print("=" * 60)

try:
    advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=test_game_id)
    
    # Check what data frames are available
    print(f"\nAvailable data frames: {len(advanced.get_data_frames())}")
    
    dfs = advanced.get_data_frames()
    for i, df in enumerate(dfs):
        print(f"\nDataFrame {i}:")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)[:10]}...")  # First 10 columns
        if not df.empty:
            print(f"  Sample row:\n{df.head(1).T}")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print(f"Error type: {type(e)}")
    
    # Try to see what's in the response
    try:
        advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=test_game_id)
        response = advanced.get_dict()
        print(f"\nAPI Response keys: {response.keys()}")
        if 'resultSets' in response:
            print(f"Number of result sets: {len(response['resultSets'])}")
            for i, rs in enumerate(response['resultSets']):
                print(f"  Result set {i}: {rs.get('name', 'unnamed')}")
    except Exception as e2:
        print(f"Could not inspect response: {e2}")

Testing BoxScoreAdvancedV2 with game_id: 0022301148

❌ Error: 'resultSet'
Error type: <class 'KeyError'>
Could not inspect response: 'resultSet'


In [47]:
from requests.exceptions import ReadTimeout
from sqlalchemy import create_engine
from dotenv import load_dotenv
import pandas as pd
import random
import time 
import json
import os

from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.static import players
from nba_api.stats.endpoints import (
    boxscoreadvancedv2,
    playergamelog
)

#---------------------------------------------------------------------------------
# Variable setup 
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Season formats - NBA API uses different formats in different endpoints
# For TeamYearByYearStats.YEAR and CommonTeamRoster.season parameter
seasons = ['2023-24', '2022-23']

# For PlayerGameLog.SEASON_ID filtering
seasons_api = ['22023', '22022']

#---------------------------------------------------------------------------------
# Get 5 test players (some well-known active players)
test_player_names = [
    'LeBron James',
    'Stephen Curry', 
    'Kevin Durant',
    'Giannis Antetokounmpo',
    'Luka Doncic'
]

all_nba_players = players.get_players()
test_players = [p for p in all_nba_players if p['full_name'] in test_player_names]

print(f"Testing with {len(test_players)} players:")
for p in test_players:
    print(f"  - {p['full_name']} (ID: {p['id']})")
print()

#------------------------------------------------------------------------------
# Gather player game logs
print("=" * 60)
print("STEP 1: Collecting basic game stats")
print("=" * 60)

player_stats_df = pd.DataFrame()

for i, player in enumerate(test_players, 1):
    while True:
        try:
            print(f"\n[{i}/{len(test_players)}] Getting stats for {player['full_name']}")
            
            # Retrieve player game stats
            game_log = playergamelog.PlayerGameLog(
                player_id=player['id'],
                season=SeasonAll.all
            )
            df = game_log.get_data_frames()[0]

            if not df.empty:
                # Filter to our target seasons using API format
                df = df[df['SEASON_ID'].isin(seasons_api)]
                if not df.empty:
                    player_stats_df = pd.concat([player_stats_df, df], ignore_index=True)
                    print(f"  ✓ Retrieved {len(df)} games from seasons {seasons}")
                else:
                    print(f"  ✗ No games in target seasons {seasons}")
            else:
                print(f"  ✗ No games found")
        
            # Rate limiting
            time.sleep(random.uniform(2, 3))
            break
            
        except (ReadTimeout, json.decoder.JSONDecodeError, Exception) as e:
            print(f"  ⚠ Error: {e} - retrying after 60 seconds")
            time.sleep(60)
            continue

print(f"\n✓ Collected basic stats: {len(player_stats_df)} total game records")

# Check if we have any data
if player_stats_df.empty:
    print("\n⚠ ERROR: No game data was collected!")
    print("This might be due to:")
    print("  - Season format mismatch")
    print("  - API changes")
    print("  - Network issues")
    print("\nExiting...")
    exit()

#-------------------------------------------------------------------------------------
# Get unique game IDs
print("\n" + "=" * 60)
print("STEP 2: Identifying unique games")
print("=" * 60)

all_game_ids = list(player_stats_df['Game_ID'].unique())
print(f"Found {len(all_game_ids)} unique games to fetch advanced stats for")

#-------------------------------------------------------------------------------------
# Fetch advanced stats
print("\n" + "=" * 60)
print("STEP 3: Fetching advanced stats")
print("=" * 60)

all_advanced_stats = pd.DataFrame()

for j, game_id in enumerate(all_game_ids, 1):
    while True:
        try:
            print(f"[{j}/{len(all_game_ids)}] Fetching advanced stats for game {game_id}")
            
            advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=game_id)
            advanced_df = advanced.get_data_frames()[0]  # Player stats
            all_advanced_stats = pd.concat([all_advanced_stats, advanced_df], ignore_index=True)
            
            # Rate limiting
            time.sleep(random.uniform(1.5, 2.5))
            break
            
        except Exception as e:
            print(f"  ⚠ Skipping game {game_id}: no advanced stats available")
            break  

print(f"\n✓ Collected advanced stats: {len(all_advanced_stats)} total records")

#-------------------------------------------------------------------------------------
# Merge basic and advanced stats
print("\n" + "=" * 60)
print("STEP 4: Merging datasets")
print("=" * 60)

combined_df = player_stats_df.merge(
    all_advanced_stats,
    on=['Game_ID', 'PLAYER_ID'],
    how='left',
    suffixes=('', '_adv')
)

print(f"✓ Merged dataset: {len(combined_df)} rows")

#-------------------------------------------------------------------------------------
# Display results
print("\n" + "=" * 60)
print("RESULTS PREVIEW")
print("=" * 60)

print("\n📊 Dataset Shape:")
print(f"  Rows: {len(combined_df)}")
print(f"  Columns: {len(combined_df.columns)}")

print("\n📋 Column Names:")
for i, col in enumerate(combined_df.columns, 1):
    print(f"  {i:3d}. {col}")

print("\n🔍 Data Types:")
print(combined_df.dtypes)

print("\n📈 Sample Records (first 3 rows):")
print(combined_df.head(3).to_string())

print("\n📊 Basic Statistics:")
print(f"  Unique players: {combined_df['PLAYER_ID'].nunique()}")
print(f"  Unique games: {combined_df['Game_ID'].nunique()}")
print(f"  Seasons covered: {sorted(combined_df['SEASON_ID'].unique())}")

print("\n✅ Null values in key columns:")
null_counts = combined_df[['E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_USG_PCT', 'E_PACE']].isnull().sum()
print(null_counts)

#-------------------------------------------------------------------------------------
# Save to database
print("\n" + "=" * 60)
print("STEP 5: Saving to database")
print("=" * 60)

table_name = "player_game_stats_test"
combined_df.to_sql(table_name, engine, if_exists='replace', index=False)

print(f"✅ Data saved to table: '{table_name}'")
print(f"   Total records: {len(combined_df)}")

# Verify save
verification_query = f"SELECT COUNT(*) as count FROM {table_name}"
verification_df = pd.read_sql(verification_query, engine)
print(f"✅ Verification: {verification_df['count'][0]} rows in database")

print("\n" + "=" * 60)
print("TEST COMPLETE!")
print("=" * 60)
print(f"\nTo inspect the table in your database, run:")
print(f"  SELECT * FROM {table_name} LIMIT 10;")

Testing with 4 players:
  - Giannis Antetokounmpo (ID: 203507)
  - Stephen Curry (ID: 201939)
  - Kevin Durant (ID: 201142)
  - LeBron James (ID: 2544)

STEP 1: Collecting basic game stats

[1/4] Getting stats for Giannis Antetokounmpo
  ✓ Retrieved 136 games from seasons ['2023-24', '2022-23']

[2/4] Getting stats for Stephen Curry
  ✓ Retrieved 130 games from seasons ['2023-24', '2022-23']

[3/4] Getting stats for Kevin Durant
  ✓ Retrieved 122 games from seasons ['2023-24', '2022-23']

[4/4] Getting stats for LeBron James
  ✓ Retrieved 126 games from seasons ['2023-24', '2022-23']

✓ Collected basic stats: 514 total game records

STEP 2: Identifying unique games
Found 496 unique games to fetch advanced stats for

STEP 3: Fetching advanced stats
[1/496] Fetching advanced stats for game 0022301148
  ⚠ Skipping game 0022301148: no advanced stats available
[2/496] Fetching advanced stats for game 0022301139
  ⚠ Skipping game 0022301139: no advanced stats available
[3/496] Fetching advan

KeyboardInterrupt: 

In [ ]:
from requests.exceptions import ReadTimeout
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import pandas as pd
import random
import time 
import json
import os

from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import (
    boxscoreadvancedv2,
    playergamelog,
    teamyearbyyearstats,
    commonteamroster
)

#---------------------------------------------------------------------------------
#define helper functions
def get_processed_players(engine, table_name='player_game_stats_temp'):
    """Get set of player IDs we've already processed"""
    inspector = inspect(engine)
    if table_name in inspector.get_table_names():
        query = f"SELECT DISTINCT PLAYER_ID FROM {table_name}"
        processed_df = pd.read_sql(query, engine)
        return set(processed_df['PLAYER_ID'].tolist())
    return set()

# def get_processed_games(engine, table_name='player_game_stats'):
#     """Get set of game IDs we've already fetched advanced stats for"""
#     inspector = inspect(engine)
#     if table_name in inspector.get_table_names():
#         # Check if the advanced stats columns exist (indicating we've processed this game)
#         query = f"SELECT DISTINCT GAME_ID FROM {table_name} WHERE E_OFF_RATING IS NOT NULL"
#         processed_df = pd.read_sql(query, engine)
#         return set(processed_df['GAME_ID'].tolist())
#     return set()

# def chunks(lst, n):
#     """Split list into chunks of size n"""
#     for i in range(0, len(lst), n):
#         yield lst[i:i + n]

#---------------------------------------------------------------------------
#Variable setup 
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Season formats - NBA API uses different formats in different endpoints!
# For TeamYearByYearStats.YEAR and CommonTeamRoster.season parameter (human-readable format)
seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20',
           '2018-19', '2017-18', '2016-17', '2015-16', '2014-15',
           '2013-14', '2012-13', '2011-12', '2010-11', '2009-10']

# For PlayerGameLog.SEASON_ID filtering (API internal format: '2YYYY')
seasons_api = ['22023', '22022', '22021', '22020', '22019',
               '22018', '22017', '22016', '22015', '22014',
               '22013', '22012', '22011', '22010', '22009']

nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total")

#----------------------------------------------------------------------------
#get players based on team rosters in each year we are targeting 

all_players = []
team_count = 0
seen_player_ids = set()

for team in nba_teams:
    team_id = team['id']
    team_name = team['full_name']
    team_count += 1
    
    print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name} (ID: {team_id})")
    
    try:
        # Check if the team was active that season
        # print(f"  Fetching season history for {team_name}...")
        # team_seasons = teamyearbyyearstats.TeamYearByYearStats(team_id=team_id)
        # seasons_df = team_seasons.get_data_frames()[0]
        
        # # Add a sleep to avoid rate limiting
        # sleep_time = random.uniform(1.5, 3.0)
        # print(f"  Sleeping for {sleep_time:.2f} seconds...")
        # time.sleep(sleep_time)
        
        for season in seasons:
                
            # Add a sleep to avoid rate limiting
            sleep_time = random.uniform(1.5, 3.0)
            print(f"  Sleeping for {sleep_time:.2f} seconds...")
            time.sleep(sleep_time)
            
            # Get team roster for that season (uses human-readable format)
            print(f"  Fetching {season} roster for {team_name}...")
            roster = commonteamroster.CommonTeamRoster(
                team_id=team_id,
                season=season
            )
            roster_df = roster.get_data_frames()[0]

            # Filter out players we've already seen
            # Convert to dict records
            # Update the set of seen player IDs
            new_players_df = roster_df[~roster_df['PLAYER_ID'].isin(seen_player_ids)]
            new_players = new_players_df.to_dict('records')
            seen_player_ids.update(new_players_df['PLAYER_ID'].tolist())
            
            player_count = len(new_players)
            all_players.extend(new_players)
            print(f"  ✓ Added {player_count} players from {team_name}")
        
    except Exception as e:
        print(f"  ⚠ Error processing {team_name}: {str(e)}")
    
    # Add a separator for readability
    print("-" * 50)
    
print("\nSummary:")
print(f"Processed {team_count} total teams")
print(f"Collected data for {len(all_players)} players")
print(len(all_players))

#------------------------------------------------------------------------------
#gather player games

# Get already processed players
print("Checking for existing progress...")
processed_player_ids = get_processed_players(engine, 'player_game_stats_temp')
print(f"Found {len(processed_player_ids)} players already processed")

# Filter out already-processed players
players_to_process = [p for p in all_players if p['PLAYER_ID'] not in processed_player_ids]
print(f"Remaining players to process: {len(players_to_process)}")

#setup dataframe
player_stats_df = pd.DataFrame()
i = 1

for player in players_to_process:
    while True:
        try:
            print(f"Getting stats for {player['PLAYER']}")
            #get player dict and ID
            PID = player['PLAYER_ID']
            
            #retreive player game stats
            game_log = playergamelog.PlayerGameLog(
                player_id=PID,
                season=SeasonAll.all
            )
            df = game_log.get_data_frames()[0]

            if not df.empty:
                # CORRECTED: Use seasons_api format for SEASON_ID filtering
                df = df[df['SEASON_ID'].isin(seasons_api)]
                if not df.empty:
                    player_stats_df = pd.concat([player_stats_df, df], ignore_index=True)
                    print(f"Data retreived for {player['PLAYER']} ({len(df)} games)")

                    # SAVE PROGRESS every 50 players
                    if i % 50 == 0:
                        print(f"\n  💾 Saving checkpoint at player {i}...")
                        player_stats_df.to_sql(
                            'player_game_stats_temp',
                            engine,
                            if_exists='append',
                            index=False
                        )
                        player_stats_df = pd.DataFrame()  # Clear memory
                        print("  ✓ Checkpoint saved\n")
                else:
                    print(f"No games in target seasons for {player['PLAYER']}")
            else:
                print(f"No games found for {player['PLAYER']}")
        
            if i % 40 == 0:
                time.sleep(round(random.uniform(60, 120), 1))
                print()
                print("Long Sleep!")
                print()
                
            else:
                time.sleep(round(random.uniform(3, 4), 1))
            i += 1

            #exit loop
            break 
            
        except (ReadTimeout, json.decoder.JSONDecodeError, Exception) as e:
            print(f"Error for {player['PLAYER']}: {e} - retrying after 60 seconds")
            time.sleep(180)
            continue 

# Save any remaining data
if not player_stats_df.empty:
    player_stats_df.to_sql('player_game_stats_temp', engine, if_exists='append', index=False)

# Load all collected player stats
print("\nLoading all collected player stats...")
all_player_stats = pd.read_sql('SELECT * FROM player_game_stats_temp', engine)


print(f"Total records: {len(all_player_stats)}")

all_player_stats.to_sql(
    'player_game_stats',
    engine,
    if_exists='replace',
    index=False
)

print("✅ Complete! All data saved to 'player_game_stats'")
print("="*60)

#-------------------------------------------------------------------------------------
#Get UNIQUE game IDs from collected data

# Extract ALL unique game IDs from the complete dataset
# all_game_ids = set(all_player_stats['GAME_ID'].unique())

# print("\nChecking which games already have advanced stats...")
# processed_game_ids = get_processed_games(engine, 'player_game_stats')
# games_to_process = list(all_game_ids - processed_game_ids)

# print(f"  Total games needed: {len(all_game_ids)}")
# print(f"  Already processed: {len(processed_game_ids)}")
# print(f"  Remaining to fetch: {len(games_to_process)}")

# if len(games_to_process) == 0:
#     print("\n✅ All games already have advanced stats! Nothing to do.")
#     exit()

# #-------------------------------------------------------------------------------------
# # define functions for batching game data and saving progress
# print("\nStep 4: Fetching advanced stats in batches...")

# game_batches = list(chunks(games_to_process, 100))
# print(f"Processing {len(games_to_process)} games in {len(game_batches)} batches")

# #-------------------------------------------------------------------------------------
# # Fetch advanced stats in batches
# j = 1

# for batch_num, batch in enumerate(game_batches, 1):
#     print(f"\n--- Batch {batch_num}/{len(game_batches)} ---")
#     batch_advanced_stats = pd.DataFrame()
    
#     for game_id in batch:
#         while True:
#             try:
#                 print(f"Fetching advanced stats for game {j}/{len(games_to_process)}: {game_id}")
                
#                 advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=game_id)
#                 advanced_df = advanced.get_data_frames()[0]  # Player stats
#                 batch_advanced_stats = pd.concat([batch_advanced_stats, advanced_df], ignore_index=True)
                
#                 # Rate limiting
#                 if j % 100 == 0:
#                     time.sleep(random.uniform(60, 120))
#                 else:
#                     time.sleep(random.uniform(1, 2))
                
#                 j += 1
#                 break
                
#             except Exception as e:
#                 print(f"Error fetching game {game_id}: {e} - retrying")
#                 time.sleep(180)
#                 continue


#     # KEY CHANGE: Merge this batch and save to final table incrementally
#     print(f"\nMerging and saving batch {batch_num}...")
    
#     # Get the player stats for games in this batch
#     batch_game_ids = batch_advanced_stats['GAME_ID'].unique()
#     batch_player_stats = all_player_stats[all_player_stats['GAME_ID'].isin(batch_game_ids)]
    
#     # Merge basic and advanced for this batch
#     batch_combined = batch_player_stats.merge(
#         batch_advanced_stats,
#         on=['GAME_ID', 'PLAYER_ID'],
#         how='left',
#         suffixes=('', '_adv')
#     )
    
#     # Save to database (append after first batch)
#     batch_combined.to_sql(
#         'player_game_stats',
#         engine,
#         if_exists='append',
#         index=False
#     )
    
#     print(f"✓ Batch {batch_num} saved ({len(batch_combined)} records)")
#     print(f"  Progress: {j-1}/{len(games_to_process)} games complete")

In [7]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.library.parameters import SeasonAll
import pandas as pd

# Get LeBron James as test player
all_players = players.get_players()
lebron = [p for p in all_players if p['full_name'] == 'LeBron James'][0]

print(f"Getting game log for {lebron['full_name']} (ID: {lebron['id']})")
print()

# Fetch all seasons
game_log = playergamelog.PlayerGameLog(
    player_id=lebron['id'],
    season=SeasonAll.all
)

df = game_log.get_data_frames()[0]

print(f"Total games returned: {len(df)}")
print()

# Show unique SEASON_ID values
print("SEASON_ID format and values:")
print("=" * 60)
unique_seasons = df['SEASON_ID'].unique()
for season in sorted(unique_seasons):
    count = len(df[df['SEASON_ID'] == season])
    print(f"  '{season}' - {count} games")

print()
print("Sample of SEASON_ID values (first 10 games):")
print(df[['GAME_DATE', 'SEASON_ID', 'MATCHUP']].head(10))

print()
print("Most recent SEASON_ID:")
most_recent = df.sort_values('GAME_DATE', ascending=False).iloc[0]
print(f"  SEASON_ID: '{most_recent['SEASON_ID']}'")
print(f"  Game Date: {most_recent['GAME_DATE']}")
print(f"  Length: {len(most_recent['SEASON_ID'])} characters")
print(f"  Last 7 chars: '{most_recent['SEASON_ID'][-7:]}'")

Getting game log for LeBron James (ID: 2544)

Total games returned: 1567

SEASON_ID format and values:
  '22003' - 79 games
  '22004' - 80 games
  '22005' - 79 games
  '22006' - 78 games
  '22007' - 75 games
  '22008' - 81 games
  '22009' - 76 games
  '22010' - 79 games
  '22011' - 62 games
  '22012' - 76 games
  '22013' - 77 games
  '22014' - 69 games
  '22015' - 76 games
  '22016' - 74 games
  '22017' - 82 games
  '22018' - 55 games
  '22019' - 67 games
  '22020' - 45 games
  '22021' - 56 games
  '22022' - 55 games
  '22023' - 71 games
  '22024' - 70 games
  '22025' - 5 games

Sample of SEASON_ID values (first 10 games):
      GAME_DATE SEASON_ID      MATCHUP
0  Dec 01, 2025     22025  LAL vs. PHX
1  Nov 28, 2025     22025  LAL vs. DAL
2  Nov 25, 2025     22025  LAL vs. LAC
3  Nov 23, 2025     22025    LAL @ UTA
4  Nov 18, 2025     22025  LAL vs. UTA
5  Apr 11, 2025     22024  LAL vs. HOU
6  Apr 09, 2025     22024    LAL @ DAL
7  Apr 08, 2025     22024    LAL @ OKC
8  Apr 06, 2025   

In [9]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import (
    teamyearbyyearstats,
    commonteamroster,
    playergamelog
)
from nba_api.stats.library.parameters import SeasonAll
import pandas as pd
import time

print("=" * 80)
print("CHECKING SEASON FORMATS FOR ALL NBA API ENDPOINTS")
print("=" * 80)

# Get Lakers as test team
nba_teams = teams.get_teams()
lakers = [t for t in nba_teams if t['full_name'] == 'Los Angeles Lakers'][0]
team_id = lakers['id']

#-----------------------------------------------------------------------------
print("\n1. TeamYearByYearStats - YEAR column format")
print("-" * 80)

team_seasons = teamyearbyyearstats.TeamYearByYearStats(team_id=team_id)
seasons_df = team_seasons.get_data_frames()[0]

print(f"Total seasons returned: {len(seasons_df)}")
print("\nYEAR column values (last 10 seasons):")
print(seasons_df[['YEAR', 'TEAM_NAME']].tail(10))
print(f"\nSample YEAR format: '{seasons_df.iloc[-1]['YEAR']}'")
print(f"Length: {len(seasons_df.iloc[-1]['YEAR'])} characters")

time.sleep(2)

#-----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("2. CommonTeamRoster - What season parameter format does it accept?")
print("-" * 80)

# Test different formats to see which works
test_formats = {
    'Format 1 (YYYY-YY)': '2023-24',
    'Format 2 (2YYYY)': '22023',
    'Format 3 (YYYY)': '2023'
}

for format_name, season_format in test_formats.items():
    try:
        print(f"\nTrying {format_name}: '{season_format}'")
        roster = commonteamroster.CommonTeamRoster(
            team_id=team_id,
            season=season_format
        )
        roster_df = roster.get_data_frames()[0]
        print(f"  ✓ SUCCESS - Returned {len(roster_df)} players")
        print(f"    Sample: {roster_df[['PLAYER', 'SEASON']].head(3).to_dict('records')}")
        
        # Check what the SEASON column returns
        if 'SEASON' in roster_df.columns:
            print(f"    SEASON column format: '{roster_df.iloc[0]['SEASON']}'")
        
        time.sleep(2)
        
    except Exception as e:
        print(f"  ✗ FAILED - {str(e)[:100]}")

#-----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("3. PlayerGameLog - SEASON_ID column format (already checked)")
print("-" * 80)
print("Format: '2YYYY' where YYYY is the year the season starts")
print("Examples:")
print("  '22024' = 2024-25 season")
print("  '22023' = 2023-24 season")
print("  '22022' = 2022-23 season")

#-----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("\nKey findings:")
print("1. TeamYearByYearStats.YEAR format: [will show above]")
print("2. CommonTeamRoster.season parameter: [will show which format works]")
print("3. PlayerGameLog.SEASON_ID format: '2YYYY' (e.g., '22023')")

CHECKING SEASON FORMATS FOR ALL NBA API ENDPOINTS

1. TeamYearByYearStats - YEAR column format
--------------------------------------------------------------------------------
Total seasons returned: 78

YEAR column values (last 10 seasons):
       YEAR TEAM_NAME
68  2016-17    Lakers
69  2017-18    Lakers
70  2018-19    Lakers
71  2019-20    Lakers
72  2020-21    Lakers
73  2021-22    Lakers
74  2022-23    Lakers
75  2023-24    Lakers
76  2024-25    Lakers
77  2025-26    Lakers

Sample YEAR format: '2025-26'
Length: 7 characters

2. CommonTeamRoster - What season parameter format does it accept?
--------------------------------------------------------------------------------

Trying Format 1 (YYYY-YY): '2023-24'
  ✓ SUCCESS - Returned 18 players
    Sample: [{'PLAYER': 'Jalen Hood-Schifino', 'SEASON': '2023'}, {'PLAYER': "D'Angelo Russell", 'SEASON': '2023'}, {'PLAYER': 'Jarred Vanderbilt', 'SEASON': '2023'}]
    SEASON column format: '2023'

Trying Format 2 (2YYYY): '22023'
  ✗ FAILE